In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [26]:
import hashlib
import json
import os
import random
from typing import Any

import torch
from auto_round import AutoRound, SignRoundConfig
from auto_round.utils.dataset_utils import CalibDataset
from datasets import load_dataset
from huggingface_hub import (
    HfApi,
    create_repo,
    get_token,
    notebook_login,
)
from transformers import AutoModelForCausalLM, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [27]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    gpu_count = torch.cuda.device_count()
    print(f"Total GPUs Detected: {gpu_count}\n")

    for idx in range(gpu_count):
        props = torch.cuda.get_device_properties(idx)
        print(f"--- GPU {idx} ---")
        print(f"  Name: {props.name}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print(f"  Total VRAM: {props.total_memory / (1024**3):.2f} GB")
        print(
            f"  Allocated VRAM: {torch.cuda.memory_allocated(idx) / (1024**3):.2f} GB"
        )
        print(
            f"  Free VRAM: {(props.total_memory - torch.cuda.memory_reserved(idx)) / (1024**3):.2f} GB\n"
        )
else:
    print("No CUDA-compatible GPUs found.")

PyTorch Version: 2.14.0+cu132
CUDA Available: True
CUDA Version: 13.2
Total GPUs Detected: 1

--- GPU 0 ---
  Name: NVIDIA L40S
  Compute Capability: 8.9
  Total VRAM: 44.39 GB
  Allocated VRAM: 5.66 GB
  Free VRAM: 38.68 GB



In [35]:
notebook_login()

In [29]:
MODEL_ID = "dots-studio/dots.mocr"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 22 files: 100%|██████████████████████| 22/22 [00:00<00:00, 660.98it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForCausalLM.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/643 [00:00<?, ?it/s]

In [8]:
model

DotsOCRForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rot

In [9]:
TARGET_NSAMPLES = 512
TARGET_SEQLEN = 4096

INDIC_LANGUAGES = (
    "Hindi",
    "Tamil",
    "Telugu",
    "Malayalam",
    "Kannada",
    "Marathi",
    "Bengali",
    "Gujarati",
    "Punjabi",
    "Odia",
    "Urdu",
    "Assamese",
)


def build_calibration_dataset(
    sources: list[dict[str, Any]],
    tokenizer,
    seqlen: int = 4096,
    total_nsamples: int = 512,
    output_jsonl: str = "./calib_mix_512_dots_mocr_real.jsonl",
    seed: int = 42,
) -> CalibDataset:
    rng = random.Random(seed)

    # 1. Resolve per-source sample quotas
    explicit_total = sum(s["num"] for s in sources if "num" in s)
    remaining_budget = max(0, total_nsamples - explicit_total)
    unassigned = [s for s in sources if "num" not in s]
    total_weight = sum(s.get("weight", 1.0) for s in unassigned) or 1.0

    quotas = []
    allocated = 0
    for idx, s in enumerate(sources):
        if "num" in s:
            q = int(s["num"])
        elif idx == len(sources) - 1:
            q = remaining_budget - allocated
        else:
            q = round(remaining_budget * (s.get("weight", 1.0) / total_weight))
            allocated += q
        quotas.append(max(1, q))

    def _passes_autoround_seqlen_check(text: str) -> tuple[bool, int]:
        """Verify exact tokenized length >= seqlen and check AutoRound's anti-repetition rule."""
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        tok_len = len(ids)
        if tok_len < seqlen:
            return False, tok_len
        window = ids[:seqlen]
        if len(window) > 1 and seqlen > 2 and window.count(window[-1]) > seqlen // 2:
            return False, tok_len
        return True, tok_len

    def _render_row(row: Any, cfg: dict[str, Any]) -> str | None:
        filter_fn = cfg.get("filter_fn")
        if filter_fn and not filter_fn(row):
            return None

        apply_tpl = cfg.get("apply_chat_template", True)

        if not isinstance(row, dict):
            return None

        fields = cfg.get("fields")

        # Two-column instruction/output pair (e.g., CohereForAI/aya_dataset: ["inputs", "targets"])
        if isinstance(fields, (list, tuple)) and len(fields) == 2 and apply_tpl:
            u_text = str(row.get(fields[0], "")).strip()
            a_text = str(row.get(fields[1], "")).strip()
            if not u_text or not a_text:
                return None
            msgs = [
                {"role": "user", "content": u_text},
                {"role": "assistant", "content": a_text},
            ]
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )

        # Single field (e.g., allenai/olmOCR-mix-1025: "natural_text")
        if isinstance(fields, str):
            raw_text = str(row.get(fields) or "").strip()
        elif isinstance(fields, (list, tuple)):
            raw_text = "\n\n".join(
                str(row.get(f) or "").strip() for f in fields if row.get(f)
            )
        else:
            raw_text = str(row.get("natural_text") or row.get("text") or "").strip()

        if not raw_text:
            return None

        if apply_tpl:
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": raw_text}],
                tokenize=False,
                add_generation_prompt=False,
            )
        return raw_text

    sep = tokenizer.eos_token or "\n\n"
    sep_tokens = len(tokenizer(sep, add_special_tokens=False)["input_ids"])

    def _digest(text: str) -> str:
        return hashlib.sha256(text.encode("utf-8")).hexdigest()

    def _collect_from_source(
        cfg: dict[str, Any],
        target_quota: int,
        run_seed: int,
        seen_hashes: set[str],
        seen_raw_hashes: set[str],
    ) -> tuple[list[str], list[int]]:
        """Stream, pack, verify >= seqlen, and deduplicate at both raw and sequence levels."""
        src = cfg["source"]
        config_name = cfg.get("config_name")
        split = cfg.get("split", "train")
        max_scan = cfg.get("max_scan", target_quota * 80)

        valid_samples: list[str] = []
        source_lengths: list[int] = []
        buf_parts: list[str] = []
        buf_tokens = 0
        scanned = 0

        def _try_add_verified(candidate_text: str) -> bool:
            h = _digest(candidate_text)
            if h in seen_hashes:
                return False
            is_valid, tok_len = _passes_autoround_seqlen_check(candidate_text)
            if is_valid:
                seen_hashes.add(h)
                valid_samples.append(candidate_text)
                source_lengths.append(tok_len)
                return True
            return False

        ds = load_dataset(src, name=config_name, split=split, streaming=True).shuffle(
            seed=run_seed, buffer_size=2000
        )

        for row in ds:
            scanned += 1
            text = _render_row(row, cfg)
            if text:
                # Deduplicate raw documents so topup never reuses an already-selected document
                raw_h = _digest(text)
                if raw_h not in seen_raw_hashes:
                    seen_raw_hashes.add(raw_h)
                    tok_len = len(
                        tokenizer(text, add_special_tokens=False)["input_ids"]
                    )
                    if tok_len >= seqlen:
                        _try_add_verified(text)
                    elif tok_len > 0:
                        buf_parts.append(text)
                        buf_tokens += tok_len + sep_tokens
                        if buf_tokens >= seqlen:
                            if _try_add_verified(sep.join(buf_parts)):
                                buf_parts, buf_tokens = [], 0

            if len(valid_samples) >= target_quota or scanned >= max_scan:
                break

        label = cfg.get("label") or (
            f"{src}:{config_name}" if config_name else str(src)
        )
        min_tok = min(source_lengths) if source_lengths else 0
        print(
            f"Source '{label}': target={target_quota}, verified={len(valid_samples)}/{target_quota} "
            f"(min_tokens={min_tok}, scanned={scanned})"
        )
        return valid_samples[:target_quota], source_lengths[:target_quota]

    # 2. Primary pass across all configured sources
    all_final_texts: list[str] = []
    verified_lengths: list[int] = []
    seen_hashes: set[str] = set()
    seen_raw_hashes: set[str] = set()
    exhausted_indices: set[int] = set()

    for idx, (cfg, quota) in enumerate(zip(sources, quotas)):
        samples, lengths = _collect_from_source(
            cfg, quota, seed + idx, seen_hashes, seen_raw_hashes
        )
        all_final_texts.extend(samples)
        verified_lengths.extend(lengths)
        if len(samples) < quota:
            exhausted_indices.add(idx)

    # 3. Automatic deduplicated topup from allenai/olmOCR-mix-1025 (`topup=True`)
    shortfall = total_nsamples - len(all_final_texts)
    if shortfall > 0:
        print(
            f"\n[Backfill] Shortfall of {shortfall} samples detected. Topping up from allenai/olmOCR-mix-1025..."
        )
        preferred_idxs = [
            i
            for i, c in enumerate(sources)
            if c.get("topup") and i not in exhausted_indices
        ]
        candidate_idxs = preferred_idxs or [
            i for i in range(len(sources)) if i not in exhausted_indices
        ]

        for idx in candidate_idxs:
            if shortfall <= 0:
                break
            extra_samples, extra_lengths = _collect_from_source(
                sources[idx], shortfall, seed + 1000 + idx, seen_hashes, seen_raw_hashes
            )
            all_final_texts.extend(extra_samples)
            verified_lengths.extend(extra_lengths)
            shortfall -= len(extra_samples)

    # 4. Assert exact count, uniqueness, and seqlen >= 4096 before saving
    all_final_texts = all_final_texts[:total_nsamples]
    verified_lengths = verified_lengths[:total_nsamples]
    assert len(all_final_texts) == len(set(all_final_texts)), "Duplicate sample found!"
    assert all(l >= seqlen for l in verified_lengths), (
        "Found sample shorter than seqlen!"
    )

    combined = list(zip(all_final_texts, verified_lengths))
    rng.shuffle(combined)
    all_final_texts = [t for t, _ in combined]
    verified_lengths = [l for _, l in combined]

    with open(output_jsonl, "w", encoding="utf-8") as out_f:
        out_f.writelines(
            json.dumps({"text": text}, ensure_ascii=False) + "\n"
            for text in all_final_texts
        )

    print(
        f"\nVerified & saved {len(all_final_texts)}/{total_nsamples} unique samples -> {output_jsonl} "
        f"(min_tokens={min(verified_lengths)}, seqlen={seqlen})"
    )
    return CalibDataset(
        output_jsonl, fields="text", apply_chat_template=False, concat=True
    )

In [10]:
calib_sources = [
    # 1. English Document OCR (`natural_text` with `primary_language == "en"`) — 224 samples
    {
        "source": "allenai/olmOCR-mix-1025",
        "config_name": "00_documents",
        "label": "allenai/olmOCR-mix-1025:00_documents (en)",
        "split": "train",
        "num": 224,
        "fields": "natural_text",
        "filter_fn": lambda r: (
            (r.get("primary_language") or "").lower().startswith("en")
        ),
        "apply_chat_template": True,
        "topup": True,
    },
    # 2. All-Language Document OCR (`natural_text` across all languages in olmOCR + deduplicated topup) — 144 samples
    {
        "source": "allenai/olmOCR-mix-1025",
        "config_name": "00_documents",
        "label": "allenai/olmOCR-mix-1025:00_documents (multilingual_topup)",
        "split": "train",
        "num": 144,
        "fields": "natural_text",
        "apply_chat_template": True,
        "topup": True,
    },
    # 3. 12 Indian Languages (`CohereForAI/aya_dataset`: 12 × 12 = 144 samples)
    *[
        {
            "source": "CohereForAI/aya_dataset",
            "label": f"CohereForAI/aya_dataset:{lang_name}",
            "split": "train",
            "num": 12,
            "fields": ["inputs", "targets"],
            "filter_fn": lambda r, target_lang=lang_name: (
                r.get("language") == target_lang
            ),
            "max_scan": 40000,
        }
        for lang_name in INDIC_LANGUAGES
    ],
]

calib_dataset = build_calibration_dataset(
    sources=calib_sources,
    tokenizer=tokenizer,
    seqlen=TARGET_SEQLEN,
    total_nsamples=TARGET_NSAMPLES,
    output_jsonl="./calib_mix_512_dots_mocr_real.jsonl",
    seed=42,
)

Source 'allenai/olmOCR-mix-1025:00_documents (en)': target=224, verified=224/224 (min_tokens=4096, scanned=1581)
Source 'allenai/olmOCR-mix-1025:00_documents (multilingual_topup)': target=144, verified=144/144 (min_tokens=4102, scanned=1798)
Source 'CohereForAI/aya_dataset:Hindi': target=12, verified=12/12 (min_tokens=4125, scanned=22294)
Source 'CohereForAI/aya_dataset:Tamil': target=12, verified=12/12 (min_tokens=4102, scanned=1156)
Source 'CohereForAI/aya_dataset:Telugu': target=12, verified=12/12 (min_tokens=4150, scanned=1046)
Source 'CohereForAI/aya_dataset:Malayalam': target=12, verified=12/12 (min_tokens=4204, scanned=8760)
Source 'CohereForAI/aya_dataset:Kannada': target=12, verified=9/12 (min_tokens=4132, scanned=40000)
Source 'CohereForAI/aya_dataset:Marathi': target=12, verified=12/12 (min_tokens=4119, scanned=10031)
Source 'CohereForAI/aya_dataset:Bengali': target=12, verified=12/12 (min_tokens=4199, scanned=9954)
Source 'CohereForAI/aya_dataset:Gujarati': target=12, verif

In [11]:
with open("./calib_mix_512_dots_mocr_real.jsonl", "r", encoding="utf-8") as f:
    lines = [json.loads(line)["text"] for line in f]

lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in lines]
double_bos = sum("<|im_start|>user\n<|im_start|>" in t for t in lines)

print(
    f"Total rows: {len(lines)} | Min tokens: {min(lengths)} | Max tokens: {max(lengths)} | Double-templated: {double_bos}"
)

Total rows: 512 | Min tokens: 4096 | Max tokens: 59534 | Double-templated: 0


In [12]:
# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1200,  # Optimization iterations per block; balances calibration time and accuracy
    gradient_accumulate_steps=4,  # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,  # Target pretrained model to be quantized
    tokenizer=tokenizer,  # Tokenizer used to process text calibration samples
    processor=processor,  # Processor for multimodal models (vision/audio), if present
    scheme="NVFP4",  # Quantization precision: 4-bit weights and 16-bit activations
    alg_configs=alg_config,  # Applies the algorithm-specific optimization settings defined above
    nsamples=512,  # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=2,  # Batch size for calibration; kept < 8 to avoid slow execution/high memory overhead
    seqlen=4096,  # Calibration context length; captures long-range dependency activations
    dataset=calib_dataset,
    low_gpu_mem_usage=False,  # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    enable_torch_compile=True,  # Compiles forward passes using torch.compile to accelerate calibration
    quant_nontext_module=False,  # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
    ignore_layers="vision_tower,lm_head",  # Excludes and visual modules from quantizationx.difference(y)
)


In [13]:
# Quantize once and export to disk across all major loader schemas
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Serializes weights to 3 formats simultaneously to guarantee compatibility:
    # - 'auto_round': Reference AutoRound format (vLLM GPTQ/Marlin, Transformers / native loader)
    # - 'auto_gptq': AutoGPTQ tensor layout (vLLM GPTQ/Marlin, TGI)
    # Use 'llm_compressor' for the compressed-tensors schema, recommended for NVFP4 inference with vLLM / Neural Magic
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)

2026-11-13 06:03:50 INFO layer_config_resolver.py L435: Ignored layers: lm_head, lm_head, vision_tower, vision_tower.blocks.[0-41].attn.proj, vision_tower.blocks.[0-41].attn.qkv, vision_tower.blocks.[0-41].mlp.fc1, vision_tower.blocks.[0-41].mlp.fc2, vision_tower.blocks.[0-41].mlp.fc3, vision_tower.merger.mlp.[0,2]
2026-11-13 06:03:50 INFO layer_config_resolver.py L435: Ignored layers: lm_head, lm_head, vision_tower, vision_tower.blocks.[0-41].attn.proj, vision_tower.blocks.[0-41].attn.qkv, vision_tower.blocks.[0-41].mlp.fc1, vision_tower.blocks.[0-41].mlp.fc2, vision_tower.blocks.[0-41].mlp.fc3, vision_tower.merger.mlp.[0,2]
2026-11-13 06:03:51 INFO base.py L1417: `torch.compile` is enabled
2026-11-13 06:03:52 INFO orchestrator.py L902: start to cache block inputs
2026-11-13 06:03:52 INFO mllm.py L88: Using MLLM template: dots_ocr
2026-11-13 06:03:52 INFO mllm.py L131: Multimodal model with non-MLLM calibration dataset './calib_mix_512_dots_mocr_real.jsonl:concat=true:fields=text' and

(DotsOCRForCausalLM(
   (model): Qwen2Model(
     (embed_tokens): Embedding(151936, 1536)
     (layers): ModuleList(
       (0-27): 28 x Qwen2DecoderLayer(
         (self_attn): Qwen2Attention(
           (q_proj): QuantLinear()
           (k_proj): QuantLinear()
           (v_proj): QuantLinear()
           (o_proj): QuantLinear()
         )
         (mlp): Qwen2MLP(
           (gate_proj): QuantLinear()
           (up_proj): QuantLinear()
           (down_proj): QuantLinear()
           (act_fn): SiLUActivation()
         )
         (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
         (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
       )
     )
     (norm): Qwen2RMSNorm((1536,), eps=1e-06)
     (rotary_emb): Qwen2RotaryEmbedding()
   )
   (lm_head): Linear(in_features=1536, out_features=151936, bias=False)
   (vision_tower): DotsVisionTransformer(
     (patch_embed): DotsViTPreprocessor(
       (patchifier): DotsPatchEmbed(
         (proj): Conv2d(3, 1536, ker

In [37]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            delete_patterns=[
                "*.safetensors",
                "*.bin",
                "*.pt",
                "*.gguf",
                "*.json",
                "*.model",
                "*.onnx",
            ],
            commit_message="Update model weights and config",
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [38]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [39]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"),
        f"{base_name}-NVFP4",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-nvfp-w4g16 to Vishva007/dots.mocr-NVFP4...


[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/dots.mocr-NVFP4
